# Compare BERT and SBERT Embeddings

This notebook compares BERT and SBERT embeddings on the Politifact and GossipCop datasets.
It includes raw-text exploratory analysis, embedding geometry, similarity structure,
projection-based visualization, and unsupervised clustering quality checks.

In [2]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, os.path.abspath('..'))

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'font.size': 11,
    'savefig.dpi': 150,
    'figure.dpi': 150,
})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def load_raw_text(dataset: str, raw_dir: str = '../data/raw') -> pd.DataFrame:
    raw_path = Path(raw_dir) / dataset / 'raw_text'
    parquet_files = sorted(raw_path.glob('*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(f'No parquet files found in {raw_path}')
    df = pd.read_parquet(parquet_files[0]).copy()
    text_col = 'text' if 'text' in df.columns else df.columns[0]
    df[text_col] = df[text_col].astype(str)
    df['word_count'] = df[text_col].str.split().str.len().fillna(0).astype(int)
    df['char_count'] = df[text_col].str.len().fillna(0).astype(int)
    df['is_empty'] = df[text_col].str.strip().eq('') | df[text_col].str.lower().eq('nan')
    df['is_duplicate'] = df[text_col].duplicated(keep=False)
    df.attrs['text_col'] = text_col
    return df


def load_embedding(dataset: str, encoder: str, interim_dir: str = '../data/interim') -> sp.spmatrix:
    emb_path = Path(interim_dir) / f'embeddings_{encoder}' / dataset / f'new_{encoder}_feature.npz'
    if not emb_path.exists():
        raise FileNotFoundError(f'Embedding file not found: {emb_path}')
    return sp.load_npz(emb_path)


def load_node_labels(dataset: str, raw_dir: str = '../data/raw') -> np.ndarray | None:
    raw_path = Path(raw_dir) / dataset
    labels_path = raw_path / 'graph_labels.npy'
    node_graph_id_path = raw_path / 'node_graph_id.npy'
    if not labels_path.exists() or not node_graph_id_path.exists():
        return None
    graph_labels = np.load(labels_path)
    node_graph_id = np.load(node_graph_id_path)
    if len(node_graph_id) == 0:
        return None
    return graph_labels[node_graph_id].astype(int)


def compute_text_statistics(df: pd.DataFrame, text_col: str) -> dict:
    text_series = df[text_col].astype(str)
    word_counts = text_series.str.split().str.len().fillna(0)
    char_counts = text_series.str.len().fillna(0)
    return {
        'n_articles': len(df),
        'n_unique_texts': text_series.nunique(),
        'duplicate_texts': int(text_series.duplicated(keep=False).sum()),
        'empty_texts': int((text_series.str.strip() == '').sum()),
        'word_count_mean': float(word_counts.mean()),
        'word_count_median': float(word_counts.median()),
        'word_count_std': float(word_counts.std(ddof=0)),
        'word_count_min': int(word_counts.min()),
        'word_count_max': int(word_counts.max()),
        'char_count_mean': float(char_counts.mean()),
        'char_count_median': float(char_counts.median()),
    }


def compute_embedding_statistics(embedding: sp.spmatrix) -> dict:
    if sp.issparse(embedding):
        dense = embedding.toarray()
        nnz = embedding.nnz
        zero_ratio = 1.0 - (nnz / (embedding.shape[0] * embedding.shape[1]))
    else:
        dense = np.asarray(embedding)
        nnz = int(np.count_nonzero(dense))
        zero_ratio = 1.0 - (nnz / dense.size)
    norms = np.linalg.norm(dense, axis=1)
    return {
        'shape': dense.shape,
        'dimensions': dense.shape[1],
        'nnz': int(nnz),
        'zero_ratio': float(zero_ratio),
        'norm_mean': float(norms.mean()),
        'norm_median': float(np.median(norms)),
        'norm_std': float(norms.std(ddof=0)),
        'dense': dense,
        'norms': norms,
    }


def sample_rows(matrix: np.ndarray, max_samples: int = 1200, seed: int = RANDOM_STATE) -> np.ndarray:
    if matrix.shape[0] <= max_samples:
        return matrix
    rng = np.random.default_rng(seed)
    indices = rng.choice(matrix.shape[0], size=max_samples, replace=False)
    return matrix[indices]


def add_mean_median_lines(ax, values) -> None:
    values = np.asarray(values)
    ax.axvline(values.mean(), color='crimson', linestyle='--', linewidth=1.8, label=f'Mean = {values.mean():.2f}')
    ax.axvline(np.median(values), color='black', linestyle=':', linewidth=1.8, label=f'Median = {np.median(values):.2f}')


def plot_text_distribution(df: pd.DataFrame, text_col: str, title: str) -> None:
    word_counts = df[text_col].astype(str).str.split().str.len().fillna(0)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(word_counts, bins=50, kde=True, ax=axes[0], color='#2E86AB')
    add_mean_median_lines(axes[0], word_counts)
    axes[0].set_title(f'{title} - Word Count Histogram')
    axes[0].set_xlabel('Word count')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    sns.boxplot(x=word_counts, ax=axes[1], color='#A5C882')
    axes[1].set_title(f'{title} - Word Count Boxplot')
    axes[1].set_xlabel('Word count')
    axes[1].set_yticks([])
    plt.tight_layout()
    plt.show()


def plot_embedding_norms(norms_dict: dict[str, np.ndarray], title: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 5))
    for label, norms in norms_dict.items():
        sns.histplot(norms, bins=50, kde=True, stat='density', element='step', fill=False, label=label, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Embedding L2 norm')
    ax.set_ylabel('Density')
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_projection(embedding: np.ndarray, labels: np.ndarray | None, title: str, method: str = 'pca', perplexity: int = 30, max_samples: int = 2000) -> None:
    dense = sample_rows(embedding, max_samples=max_samples)
    if labels is not None:
        labels = sample_rows(labels.reshape(-1, 1), max_samples=max_samples).ravel()
    if method == 'pca':
        coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(dense)
    else:
        perplexity = min(perplexity, max(5, dense.shape[0] - 1))
        coords = TSNE(n_components=2, perplexity=perplexity, init='pca', learning_rate='auto', random_state=RANDOM_STATE).fit_transform(dense)
    fig, ax = plt.subplots(figsize=(8, 6))
    if labels is None:
        ax.scatter(coords[:, 0], coords[:, 1], s=12, alpha=0.7, color='#7B2CBF')
    else:
        scatter = ax.scatter(coords[:, 0], coords[:, 1], c=labels, cmap='coolwarm', s=12, alpha=0.75)
        fig.colorbar(scatter, ax=ax, label='Label')
    ax.set_title(title)
    ax.set_xlabel(f'{method.upper()} 1')
    ax.set_ylabel(f'{method.upper()} 2')
    plt.tight_layout()
    plt.show()


def evaluate_clustering(embedding: np.ndarray, sample_size: int = 3000) -> tuple[float, np.ndarray]:
    dense = sample_rows(embedding, max_samples=sample_size)
    clusters = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10).fit_predict(dense)
    score = silhouette_score(dense, clusters) if len(np.unique(clusters)) > 1 else float('nan')
    return score, clusters

## Raw Text EDA

The raw article text is the first sanity check for both datasets. We inspect article counts, word-count distributions, duplicates, and empty text entries before moving to embedding-space analysis.

In [ ]:
datasets = ['politifact', 'gossipcop']
raw_frames = {}
raw_stats_rows = []

for dataset in datasets:
    df = load_raw_text(dataset)
    text_col = df.attrs['text_col']
    raw_frames[dataset] = df
    stats = compute_text_statistics(df, text_col)
    stats['dataset'] = dataset
    raw_stats_rows.append(stats)

raw_stats_df = pd.DataFrame(raw_stats_rows).set_index('dataset')
display(raw_stats_df[['n_articles', 'n_unique_texts', 'duplicate_texts', 'empty_texts', 'word_count_mean', 'word_count_median', 'word_count_std']].round(2))

for dataset, df in raw_frames.items():
    text_col = df.attrs['text_col']
    print(f'\n{dataset.upper()} - sample rows')
    display(df[[text_col, 'word_count', 'char_count', 'is_empty', 'is_duplicate']].head(3))
    plot_text_distribution(df, text_col, dataset.upper())

## Embedding Comparison

This section compares BERT and SBERT in embedding space. We inspect sparsity, norms, cosine similarity, PCA, t-SNE, and simple clustering quality as an unsupervised proxy for class separation.

In [ ]:
embedding_rows = []
embedding_cache = {}
label_cache = {}

for dataset in datasets:
    labels = load_node_labels(dataset)
    label_cache[dataset] = labels
    for encoder in ['bert', 'sbert']:
        embedding = load_embedding(dataset, encoder)
        stats = compute_embedding_statistics(embedding)
        embedding_cache[(dataset, encoder)] = stats
        embedding_rows.append({
            'dataset': dataset,
            'encoder': encoder.upper(),
            'shape': stats['shape'],
            'dimensions': stats['dimensions'],
            'nnz': stats['nnz'],
            'zero_ratio': stats['zero_ratio'],
            'norm_mean': stats['norm_mean'],
            'norm_median': stats['norm_median'],
            'norm_std': stats['norm_std'],
        })

embedding_stats_df = pd.DataFrame(embedding_rows)
display(embedding_stats_df.round(4))

for dataset in datasets:
    plot_embedding_norms(
        {
            'BERT': embedding_cache[(dataset, 'bert')]['norms'],
            'SBERT': embedding_cache[(dataset, 'sbert')]['norms'],
        },
        f'{dataset.upper()} - Embedding Norm Distribution'
    )

similarity_rows = []
for dataset in datasets:
    for encoder in ['bert', 'sbert']:
        dense = embedding_cache[(dataset, encoder)]['dense']
        sampled = sample_rows(dense, max_samples=500)
        similarity = cosine_similarity(sampled)
        upper = similarity[np.triu_indices_from(similarity, k=1)]
        similarity_rows.append({
            'dataset': dataset,
            'encoder': encoder.upper(),
            'avg_similarity': float(upper.mean()),
            'std_similarity': float(upper.std(ddof=0)),
            'min_similarity': float(upper.min()),
            'max_similarity': float(upper.max()),
        })
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.histplot(upper, bins=50, kde=True, ax=ax, color='#34495E')
        ax.set_title(f'{dataset.upper()} - {encoder.upper()} Pairwise Cosine Similarity')
        ax.set_xlabel('Cosine similarity')
        ax.set_ylabel('Frequency')
        plt.tight_layout()
        plt.show()

similarity_df = pd.DataFrame(similarity_rows)
display(similarity_df.round(4))

cluster_rows = []
for dataset in datasets:
    labels = label_cache[dataset]
    for encoder in ['bert', 'sbert']:
        dense = embedding_cache[(dataset, encoder)]['dense']
        sampled = sample_rows(dense, max_samples=3000)
        sampled_labels = None if labels is None else sample_rows(labels.reshape(-1, 1), max_samples=3000).ravel()
        silhouette, cluster_labels = evaluate_clustering(sampled, sample_size=3000)
        cluster_rows.append({
            'dataset': dataset,
            'encoder': encoder.upper(),
            'silhouette_score': silhouette,
            'cluster_0_count': int((cluster_labels == 0).sum()),
            'cluster_1_count': int((cluster_labels == 1).sum()),
        })
        plot_projection(dense, sampled_labels, f'{dataset.upper()} - {encoder.upper()} PCA Projection', method='pca', max_samples=2000)
        plot_projection(dense, sampled_labels, f'{dataset.upper()} - {encoder.upper()} t-SNE Projection', method='tsne', perplexity=30, max_samples=1000)

cluster_df = pd.DataFrame(cluster_rows)
display(cluster_df.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=cluster_df, x='encoder', y='silhouette_score', hue='dataset', ax=ax, palette='Set2')
ax.set_title('Silhouette Score Comparison')
ax.set_xlabel('Encoder')
ax.set_ylabel('Silhouette score')
plt.tight_layout()
plt.show()

## Interpretation

The word-count histogram describes the length distribution of the article corpus. A right-skewed distribution means most articles are short, while a long tail indicates a smaller number of long-form articles that may behave differently under graph construction and embedding learning.

Cosine similarity measures angular closeness between vectors. If one encoder yields a lower average similarity and a broader similarity distribution, it is usually preserving more semantic variation across articles.

PCA reveals the dominant variance directions, so visible separation there suggests a strong coarse-grained signal. t-SNE emphasizes local neighborhoods, which is useful for seeing whether articles with the same label form tighter subclusters.

The silhouette score gives an unsupervised summary of cluster separation. When SBERT scores better than BERT, that is evidence that sentence-level semantic embeddings are better aligned with the binary fake-news discrimination task.